# aither-kvcache: vLLM Quickstart

This notebook walks you through using **aither-kvcache** with vLLM to compress your KV cache to 4-bit, giving you **3.8x more context** for the same VRAM budget.

**What you'll learn:**
1. Install and verify the package
2. Run standalone quantization (no vLLM needed)
3. Integrate with vLLM using hooks (recommended)
4. Use PRIMARY mode for maximum VRAM savings
5. Graph-aware eviction for smarter cache management

**Requirements:** Python 3.10+, PyTorch 2.0+, CUDA GPU (Ampere or newer recommended)

## 1. Installation

In [ ]:
# Install with vLLM integration + Triton GPU kernels
!pip install aither-kvcache[all]

In [ ]:
import torch
import aither_kvcache

print(f"aither-kvcache v{aither_kvcache.__version__}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
    print(f"VRAM: {vram:.1f} GB")

## 2. Standalone Quantization

Before touching vLLM, let's verify TurboQuant works correctly on your hardware.
This encodes random KV vectors to 4-bit and checks that MSE stays within the
paper's theoretical bounds.

In [ ]:
from aither_kvcache import TurboQuant

device = "cuda" if torch.cuda.is_available() else "cpu"
tq = TurboQuant(head_dim=128, bits=4, device=device)

# Validate correctness against information-theoretic bounds
result = tq.validate(num_vectors=50000)

print(f"MSE:              {result['mse']:.6f}")
print(f"Theory bounds:    [{result['mse_theory_lower']:.6f}, {result['mse_theory_upper']:.6f}]")
print(f"Ratio to lower:   {result['mse_ratio_to_lower']:.2f}x (paper claims <= 2.7x)")
print(f"Compression:      {result['compression_vs_fp16']:.1f}x vs FP16, {result['compression_vs_fp8']:.1f}x vs FP8")
print(f"Engine:           {'Triton' if result['triton_active'] else 'PyTorch'}")

In [ ]:
# Encode/decode a batch of KV vectors
kv_vectors = torch.randn(32, 8, 128, device=device, dtype=torch.float16)  # [batch, heads, dim]

packed, norms = tq.encode(kv_vectors)
decoded = tq.decode(packed, norms)

mse = ((kv_vectors.float() - decoded.float()) ** 2).mean().item()
print(f"Input shape:  {kv_vectors.shape}")
print(f"Packed shape: {packed.shape} (dtype={packed.dtype})")
print(f"Norms shape:  {norms.shape}")
print(f"Output shape: {decoded.shape}")
print(f"Round-trip MSE: {mse:.6f}")

## 3. Memory Savings Calculator

See exactly how much VRAM you'll save for your model and context length.

In [ ]:
# Common model configurations: (name, num_layers, num_kv_heads, head_dim)
models = [
    ("Llama 3.1 8B",     32, 8,  128),
    ("Mistral 7B v0.3",  32, 8,  128),
    ("Qwen2.5 14B",      40, 8,  128),
    ("Llama 3.1 70B",    80, 8,  128),
    ("Qwen2.5 72B",      80, 8,  128),
]

seq_len = 32768  # Change this to your target context length

print(f"KV Cache Memory at {seq_len:,} tokens\n")
print(f"{'Model':<20} {'FP16':>8} {'FP8':>8} {'TQ4':>8} {'TQ3':>8} {'TQ2':>8}")
print("-" * 62)

for name, layers, kv_heads, dim in models:
    vecs_per_token = 2 * layers * kv_heads  # K + V
    fp16 = vecs_per_token * dim * 2 * seq_len / (1024**2)        # 2 bytes/dim
    fp8  = vecs_per_token * dim * 1 * seq_len / (1024**2)        # 1 byte/dim
    tq4  = vecs_per_token * 68  * seq_len / (1024**2)            # 68 bytes/vec
    tq3  = vecs_per_token * 52  * seq_len / (1024**2)            # 52 bytes/vec
    tq2  = vecs_per_token * 36  * seq_len / (1024**2)            # 36 bytes/vec
    print(f"{name:<20} {fp16:>6.1f}GB {fp8:>6.1f}GB {tq4:>6.1f}GB {tq3:>6.1f}GB {tq2:>6.1f}GB")

## 4. Throughput Benchmark

Measure encode/decode throughput on your GPU.

In [ ]:
for bits in [4, 3, 2]:
    tq = TurboQuant(head_dim=128, bits=bits, device=device)
    result = tq.benchmark(num_vectors=32768)
    print(f"[{bits}-bit] Encode: {result['encode_us']:.0f} us ({result['encode_throughput_mvec_s']:.2f} Mvec/s)  "
          f"Decode: {result['decode_us']:.0f} us ({result['decode_throughput_mvec_s']:.2f} Mvec/s)")

## 5. vLLM Integration — Hook Mode (Recommended)

The hook-based approach patches vLLM's Triton attention in-place.
It preserves `torch.compile` + CUDA graphs for maximum throughput.

### Option A: Environment variables (simplest)

```bash
# Just set these before starting vLLM:
export AITHER_TQ_MODE=tq4-primary
export AITHER_TQ_BITS=4
export AITHER_TQ_FUSED=1
export AITHER_TQ_FORCE_TRITON=1  # Required on Blackwell (SM_100+)

# Then start vLLM normally:
vllm serve meta-llama/Llama-3.1-8B-Instruct \
    --attention-backend TRITON_ATTN \
    --compilation-config '{"cudagraph_mode":"piecewise"}'
```

### Option B: Programmatic (full control)

In [ ]:
# This cell shows the programmatic approach.
# Run this BEFORE starting the vLLM engine.

import os
os.environ["AITHER_TQ_MODE"] = "tq4-primary"
os.environ["AITHER_TQ_BITS"] = "4"
os.environ["AITHER_TQ_FUSED"] = "1"

# Step 1: Apply engine patches (BEFORE vLLM starts)
from aither_kvcache.vllm import apply_tq_patches
patched = apply_tq_patches(bits=4)
print(f"Engine patches applied: {patched}")

# Step 2: Start vLLM
# from vllm import LLM
# llm = LLM(model="meta-llama/Llama-3.1-8B-Instruct", gpu_memory_utilization=0.90)

# Step 3: Apply hooks (AFTER model loads)
from aither_kvcache.vllm import apply_tq_hooks
# apply_tq_hooks()  # Uncomment after LLM() call above

print("\nReady! Uncomment the LLM() and apply_tq_hooks() lines to run.")
print("The KV cache will be TQ-compressed automatically — no code changes needed.")

### Option C: sitecustomize (zero code changes)

Copy the provided `sitecustomize.py` so TQ patches apply automatically when vLLM imports:

```bash
# Find the sitecustomize hook:
python -c "from aither_kvcache.vllm import sitecustomize; print(sitecustomize.__file__)"

# Copy it to your PYTHONPATH:
cp /path/to/sitecustomize.py ./sitecustomize.py
export PYTHONPATH=".:$PYTHONPATH"

# Start vLLM — TQ hooks apply automatically:
vllm serve meta-llama/Llama-3.1-8B-Instruct --attention-backend TRITON_ATTN
```

## 6. Full vLLM Example (end-to-end)

Complete working example — copy this to a script and run it.

In [ ]:
EXAMPLE_SCRIPT = '''
#!/usr/bin/env python3
"""aither-kvcache + vLLM: complete working example."""
import os
os.environ["AITHER_TQ_MODE"] = "tq4-primary"
os.environ["AITHER_TQ_BITS"] = "4"
os.environ["AITHER_TQ_FUSED"] = "1"
# os.environ["AITHER_TQ_FORCE_TRITON"] = "1"  # Uncomment for Blackwell GPUs

# 1. Patch vLLM engine BEFORE import
from aither_kvcache.vllm import apply_tq_patches, apply_tq_hooks
apply_tq_patches(bits=4)

# 2. Create vLLM engine
from vllm import LLM, SamplingParams
llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    gpu_memory_utilization=0.90,
    # enforce_eager=False,  # Default: compile + CUDA graphs
)

# 3. Apply TQ hooks AFTER model loads
apply_tq_hooks()

# 4. Use it — KV cache is now TQ-compressed automatically
prompts = [
    "Explain quantum computing in simple terms.",
    "Write a Python function to find prime numbers.",
    "What are the main differences between TCP and UDP?",
]
params = SamplingParams(temperature=0.7, max_tokens=256)
outputs = llm.generate(prompts, params)

for output in outputs:
    print(f"Prompt: {output.prompt[:60]}...")
    print(f"Output: {output.outputs[0].text[:200]}")
    print()
'''

print(EXAMPLE_SCRIPT)

## 7. Hybrid Modes (tq35 / tq25)

Split-group quantization for better quality at the same compression ratio.
Uses variance-based dimension splitting + QJL residual encoding.

In [ ]:
from aither_kvcache import HybridTurboQuant

# tq35: 3.5 average bits (50% @ 4-bit + 50% @ 3-bit)
htq = HybridTurboQuant(head_dim=128, mode="tq35", device=device)
htq.calibrate_uniform()

kv = torch.randn(16, 8, 128, device=device, dtype=torch.float16)
packed = htq.encode(kv)
decoded = htq.decode(packed)

mse = ((kv.float() - decoded.float()) ** 2).mean().item()
print(f"Mode:        tq35 (3.5 avg bits)")
print(f"Packed dim:  {packed.shape[-1]} (from {kv.shape[-1]})")
print(f"Compression: {htq.compression_ratio():.2f}x vs FP16")
print(f"MSE:         {mse:.6f}")

# Compare with uniform 4-bit
tq4 = TurboQuant(head_dim=128, bits=4, device=device)
packed4, norms4 = tq4.encode(kv)
decoded4 = tq4.decode(packed4, norms4)
mse4 = ((kv.float() - decoded4.float()) ** 2).mean().item()
print(f"\nUniform 4-bit MSE: {mse4:.6f}")
print(f"tq35 gets {mse4/mse:.1f}x better MSE at only {htq.compression_ratio()/tq4.compression_ratio():.0%} the compression")

## 8. Graph-Aware Eviction

Standard eviction (LRU/FIFO) doesn't know your system prompt from throwaway tokens.
The `KVCacheGraph` builds a relationship graph over cache blocks so eviction is
structurally informed — system prompt blocks are protected, frequently co-attended
blocks stay together, and cold-tier prefetch is guided by graph neighbors.

In [ ]:
from aither_kvcache import KVCacheGraph, GraphEvictionAdvisor, EdgeType

# Create graph — system prompt blocks are never evicted
graph = KVCacheGraph(protected_sources={"system", "tools"})

# Simulate blocks entering the KV cache
graph.add_block(0, "system",    importance=0.95, token_range=(0, 16))
graph.add_block(1, "system",    importance=0.90, token_range=(16, 32))
graph.add_block(2, "tools",     importance=0.85, token_range=(32, 48))
graph.add_block(3, "user",      importance=0.60, token_range=(48, 64))
graph.add_block(4, "assistant", importance=0.30, token_range=(64, 80))
graph.add_block(5, "user",      importance=0.50, token_range=(80, 96))
graph.add_block(6, "assistant", importance=0.25, token_range=(96, 112))

# Feed attention patterns — edges form automatically
graph.on_attention_step([0, 1, 2, 3, 4])  # First decode step
graph.on_attention_step([0, 1, 2, 5, 6])  # Second decode step
graph.on_temporal_sequence([4, 5, 6])      # Sequential generation

# Ask who to evict — system/tools blocks are structurally protected
victims = graph.suggest_eviction(n_blocks=2)
print(f"Eviction candidates: blocks {victims}")
print(f"(System blocks 0, 1, 2 are protected — never evicted)")

stats = graph.get_stats()
print(f"\nGraph stats: {stats['num_nodes']} nodes, {stats['num_edges']} edges")

In [ ]:
# Background advisor — zero overhead on the decode path
advisor = GraphEvictionAdvisor(graph, interval=0.5, max_stale=2.0)
advisor.start()

import time
time.sleep(0.6)  # Let the advisor compute its first ranking

# These calls are lock-free — safe to call from the hot decode path
candidates = advisor.get_eviction_candidates(n=3)
print(f"Pre-computed eviction candidates: {candidates}")

prefetch = advisor.get_prefetch_candidates([0, 1], n=4)
print(f"Prefetch suggestions: {prefetch}")

print(f"Advisor stats: {advisor.get_stats()}")

advisor.stop()
print("Advisor stopped.")

## 9. Zero-Buffer Fused Attention

Compute attention directly from TQ-compressed data — no decompression buffer.
The math works in the rotated domain: rotate the query forward once, compute
attention against codebook-decoded values, rotate back once.

In [ ]:
from aither_kvcache.fused_attention import TQPagedAttention

tq = TurboQuant(head_dim=128, bits=4, device=device)
attn = TQPagedAttention(tq, num_query_heads=32)

# Simulate a paged KV cache with 4 blocks of 16 tokens each
batch_size = 1
num_kv_heads = 8
block_size = 16
num_blocks = 4
packed_dim = 64  # 128 dims / 2 nibbles per byte for 4-bit

query = torch.randn(batch_size, 32, 128, device=device, dtype=torch.float16)

# Pre-compressed KV blocks (in practice, vLLM fills these)
k_packed = torch.randint(0, 255, (num_blocks, block_size, num_kv_heads, packed_dim),
                         device=device, dtype=torch.uint8)
k_norms = torch.randn(num_blocks, block_size, num_kv_heads, device=device)
v_packed = torch.randint(0, 255, (num_blocks, block_size, num_kv_heads, packed_dim),
                         device=device, dtype=torch.uint8)
v_norms = torch.randn(num_blocks, block_size, num_kv_heads, device=device)

block_tables = torch.tensor([[0, 1, 2, 3]], device=device, dtype=torch.int32)
context_lens = torch.tensor([64], device=device, dtype=torch.int32)

output = attn.forward(query, k_packed, k_norms, v_packed, v_norms,
                       block_tables, context_lens, block_size=block_size)

print(f"Query shape:  {query.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention computed directly from {num_blocks * block_size} compressed KV tokens")
print(f"No decompression buffer allocated!")

## 10. Environment Variable Reference

| Variable | Default | Description |
|----------|---------|-------------|
| `AITHER_TQ_MODE` | (none) | `tq4-primary`, `tq3-primary`, `tq35`, `tq25` |
| `AITHER_TQ_BITS` | `4` | Bit width: 2, 3, or 4 |
| `AITHER_TQ_FUSED` | `0` | `1` = fused Triton decode, `0` = standard |
| `AITHER_TQ_EAGER` | `0` | `1` = disable torch.compile (debugging) |
| `AITHER_TQ_FORCE_TRITON` | `0` | `1` = force Triton on Blackwell (SM_100+) |
| `AITHER_TQ_PRIMARY` | `0` | `1` = PRIMARY mode (legacy, use `_MODE` instead) |
| `AITHER_TQ_DEBUG` | `0` | `1` = verbose per-step logging to stderr |
| `AITHER_TQ_DEBUG_STEPS` | `10` | Number of steps to log per layer |

## Next Steps

- **Docs**: [GitHub README](https://github.com/Aitherium/aitherkvcache)
- **Paper**: [arXiv:2504.19874](https://arxiv.org/abs/2504.19874) (Zandieh et al.)
- **Questions**: [GitHub Discussions](https://github.com/Aitherium/aitherkvcache/discussions)
- **Bugs**: [GitHub Issues](https://github.com/Aitherium/aitherkvcache/issues)